# DHE (Deep Hash Embedding) — Criteo Benchmark
**Paper:** "Learning to Embed Categorical Features without Embedding Tables for Recommendation" — Google, KDD 2021

## Architecture Overview
- **DHE**: Thay thế lookup table bằng một MLP nhỏ ánh xạ hash-code → embedding vector 64 chiều
- **DeepFM & DCN**: 13 integer features → `log1p` normalize; 26 cat features → DHE → 64-dim
- **DLRM**: 13 integer features → MLP → 64-dim; 26 cat features → DHE → 64-dim
- **Loss**: BCEWithLogitsLoss (model output raw logits, KHÔNG sigmoid bên trong)
- **Metric**: AUC-ROC

In [1]:
# ============================================================
# Cell 1 — Install & Imports
# ============================================================
import os, math, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

print(f'PyTorch: {torch.__version__}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

PyTorch: 2.10.0+cu128
Device : cuda


In [2]:
# ============================================================
# Cell 2 — CONFIG  ← Chỉnh ở đây để test nhanh hoặc full run
# ============================================================
CFG = {
    # --- Data ---
    'data_dir'    : '/kaggle/input/datasets/huy291/criteo-cleaned-data/data',  # thư mục parquet
    'num_files'   : 50,         
    'val_ratio'   : 0.1,        # tỉ lệ validation
    'test_ratio'  : 0.1,        # tỉ lệ test

    # --- DHE ---
    'embed_dim'   : 64,         # chiều output embedding
    'dhe_num_hash': 1024,       # K: số hash functions (dimension of hash code)
    'dhe_hidden'  : [512, 256], # hidden layers của DHE MLP

    # --- Integer features (cho DLRM) ---
    'int_mlp_dims': [64, 64],   # MLP embed integer → 64 dim

    # --- Training ---
    'batch_size'  : 4096,
    'epochs'      : 5,
    'lr'          : 1e-3,
    'weight_decay': 1e-5,

    # --- Models to run ---
    # DeepFM params
    'deepfm_hidden': [400, 400, 400],
    # DCN params
    'dcn_cross_layers': 3,
    'dcn_hidden'  : [512, 512],
    # DLRM params
    'dlrm_bottom_mlp': [256, 128, 64],
    'dlrm_top_mlp'   : [512, 256, 128],
}

DENSE_COLS = [f'I{i}' for i in range(1, 14)]
CAT_COLS   = [f'C{i}' for i in range(1, 27)]
LABEL_COL  = 'label'
print('Config loaded.')

Config loaded.


In [3]:
# ============================================================
# Cell 3 — Load & Preprocess Data
# ============================================================
def load_data(data_dir, num_files=None):
    files = sorted([f for f in os.listdir(data_dir) if f.endswith('.parquet')])
    if num_files is not None:
        files = files[:num_files]
    print(f'Loading {len(files)} parquet files...')
    dfs = []
    for i, f in enumerate(files):
        df = pd.read_parquet(os.path.join(data_dir, f))
        dfs.append(df)
        if (i+1) % 5 == 0:
            print(f'  {i+1}/{len(files)} files loaded, shape so far: {pd.concat(dfs).shape}')
    df = pd.concat(dfs, ignore_index=True)
    print(f'Total shape: {df.shape}')
    return df

df = load_data(CFG['data_dir'], CFG['num_files'])
print(df.head(2))

Loading 50 parquet files...
  5/50 files loaded, shape so far: (4418151, 40)
  10/50 files loaded, shape so far: (8835527, 40)
  15/50 files loaded, shape so far: (13798736, 40)
  20/50 files loaded, shape so far: (18213880, 40)
  25/50 files loaded, shape so far: (23184883, 40)
  30/50 files loaded, shape so far: (27603094, 40)
  35/50 files loaded, shape so far: (32013306, 40)
  40/50 files loaded, shape so far: (36982531, 40)
  45/50 files loaded, shape so far: (41401166, 40)
  50/50 files loaded, shape so far: (45840617, 40)
Total shape: (45840617, 40)
   label  I1  I2  I3  I4    I5  I6  I7  I8   I9  ...       C17       C18  \
0      0   1   1   5   0  1382   4  15   2  181  ...  e5ba7672  f54016b9   
1      0   2   0  44   1   102   8   2   2    4  ...  07c540c4  b04e4670   

        C19       C20       C21      C22       C23       C24       C25  \
0  21ddcdc9  b1252a9d  07b5194c  unknown  3a171ecb  c5c50484  e8b83407   
1  21ddcdc9  5840adea  60f6221e  unknown  3a171ecb  43f13e8b

In [4]:
# ============================================================
# Cell 4 — Label Encode categorical + int dtype
# ============================================================
# Encode cat cols → int index (0-based)
vocab_sizes = []
for col in CAT_COLS:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    vocab_sizes.append(int(df[col].max()) + 1)

# Integer cols: đã điền mean sẵn, chỉ cần ép float
df[DENSE_COLS] = df[DENSE_COLS].astype(np.float32)
df[LABEL_COL]  = df[LABEL_COL].astype(np.float32)

print('Vocab sizes (first 5):', vocab_sizes[:5])
print('Dense sample:', df[DENSE_COLS].head(2).values)

Vocab sizes (first 5): [1460, 583, 10131227, 2202608, 305]
Dense sample: [[1.000e+00 1.000e+00 5.000e+00 0.000e+00 1.382e+03 4.000e+00 1.500e+01
  2.000e+00 1.810e+02 1.000e+00 2.000e+00 0.000e+00 2.000e+00]
 [2.000e+00 0.000e+00 4.400e+01 1.000e+00 1.020e+02 8.000e+00 2.000e+00
  2.000e+00 4.000e+00 1.000e+00 1.000e+00 0.000e+00 4.000e+00]]


In [5]:
# ============================================================
# Cell 5 — Train / Val / Test Split
# ============================================================
from sklearn.model_selection import train_test_split

n = len(df)
idx = np.arange(n)
test_size  = int(n * CFG['test_ratio'])
val_size   = int(n * CFG['val_ratio'])
train_size = n - test_size - val_size

# Giữ thứ tự thời gian: train → val → test
train_df = df.iloc[:train_size].reset_index(drop=True)
val_df   = df.iloc[train_size:train_size+val_size].reset_index(drop=True)
test_df  = df.iloc[train_size+val_size:].reset_index(drop=True)

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

Train: 36,672,495  Val: 4,584,061  Test: 4,584,061


In [6]:
# ============================================================
# Cell 6 — Dataset
# ============================================================
class CriteoDataset(Dataset):
    def __init__(self, df, dense_cols, cat_cols, label_col):
        self.dense = torch.tensor(df[dense_cols].values, dtype=torch.float32)
        self.cat   = torch.tensor(df[cat_cols].values,   dtype=torch.long)
        self.label = torch.tensor(df[label_col].values,  dtype=torch.float32)

    def __len__(self):
        return len(self.label)

    def __getitem__(self, idx):
        return self.dense[idx], self.cat[idx], self.label[idx]

train_ds = CriteoDataset(train_df, DENSE_COLS, CAT_COLS, LABEL_COL)
val_ds   = CriteoDataset(val_df,   DENSE_COLS, CAT_COLS, LABEL_COL)
test_ds  = CriteoDataset(test_df,  DENSE_COLS, CAT_COLS, LABEL_COL)

train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size']*2, shuffle=False, num_workers=4, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=CFG['batch_size']*2, shuffle=False, num_workers=4, pin_memory=True)
print('DataLoaders ready.')

DataLoaders ready.


In [7]:
# ============================================================
# Cell 7 — DHE Module
# ============================================================
# Nguồn: "Learning to Embed Categorical Features without Embedding Tables" KDD 2021
# Ý tưởng: với mỗi category ID, tạo hash code K chiều bằng K hàm hash độc lập
# rồi đưa qua MLP để ra embedding 64 chiều.
# Không dùng bảng lookup → tiết kiệm memory với vocab lớn, tổng quát hoá tốt hơn.

class DHEEncoder(nn.Module):
    """
    Deep Hash Embedding cho một feature với vocab_size tùy ý.
    Input : (B,) LongTensor các category ID
    Output: (B, embed_dim) FloatTensor
    """
    def __init__(self, vocab_size: int, embed_dim: int, num_hashes: int = 1024, hidden_dims=(512, 256)):
        super().__init__()
        self.vocab_size = vocab_size
        self.num_hashes = num_hashes
        self.embed_dim  = embed_dim

        # Random prime numbers dùng làm hash seeds — cố định để stable
        # Dùng CRC-style universal hashing: h_i(x) = ((a_i * x + b_i) mod p) mod num_hashes
        # Để tránh collisions, dùng large prime p
        torch.manual_seed(42)
        self.register_buffer('a_vec', torch.randint(1, 2**31 - 1, (num_hashes,), dtype=torch.long))
        self.register_buffer('b_vec', torch.randint(0, 2**31 - 1, (num_hashes,), dtype=torch.long))
        self.p = 2**31 - 1  # Mersenne prime

        # MLP: num_hashes → embed_dim
        # Input sau hash: binary ±1 encoding
        layers = []
        in_d = num_hashes
        for h in hidden_dims:
            layers += [nn.Linear(in_d, h), nn.LayerNorm(h), nn.GELU()]
            in_d = h
        layers.append(nn.Linear(in_d, embed_dim))
        self.mlp = nn.Sequential(*layers)

    def _hash_encode(self, ids: torch.Tensor) -> torch.Tensor:
        """
        ids: (B,) long
        returns: (B, K) float in {-1, +1}
        """
        B = ids.size(0)
        # (B, K)
        ids_exp = ids.unsqueeze(1).expand(B, self.num_hashes)        # (B, K)
        # Universal hash
        h = ((self.a_vec * ids_exp + self.b_vec) % self.p) % 2      # (B, K) in {0,1}
        return h.float() * 2 - 1  # map to {-1, +1}

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        code = self._hash_encode(ids)  # (B, K)
        return self.mlp(code)          # (B, embed_dim)


class DHELayer(nn.Module):
    """
    Wrapper áp dụng DHE cho tất cả 26 cat features song song.
    Input : (B, 26) LongTensor
    Output: (B, 26, embed_dim)
    """
    def __init__(self, vocab_sizes, embed_dim, num_hashes, hidden_dims):
        super().__init__()
        self.encoders = nn.ModuleList([
            DHEEncoder(v, embed_dim, num_hashes, hidden_dims)
            for v in vocab_sizes
        ])
        self.embed_dim = embed_dim

    def forward(self, cat_x: torch.Tensor) -> torch.Tensor:
        # cat_x: (B, 26)
        embs = [enc(cat_x[:, i]) for i, enc in enumerate(self.encoders)]  # list of (B, D)
        return torch.stack(embs, dim=1)  # (B, 26, D)

print('DHE module defined.')

DHE module defined.


In [8]:
# ============================================================
# Cell 8 — Numeric Feature Processors
# ============================================================

class Log1pNormalizer(nn.Module):
    """
    Chuẩn hóa integer features bằng log1p cho DeepFM & DCN.
    Áp dụng: x_norm = log(1 + |x|) * sign(x)  (handle negative values)
    Sau đó LayerNorm để ổn định training.
    """
    def __init__(self, num_features):
        super().__init__()
        self.norm = nn.LayerNorm(num_features)

    def forward(self, x):
        x_log = torch.log1p(torch.abs(x)) * torch.sign(x)
        return self.norm(x_log)


class DenseEmbedMLP(nn.Module):
    """
    Embed 13 integer features thành 13 x embed_dim cho DLRM.
    Mỗi feature được embed riêng qua cùng 1 MLP (shared weights).
    Input : (B, 13)
    Output: (B, 13, embed_dim)
    Tại sao shared? — tất cả int features đều là count features, cùng scale sau log1p,
    dùng shared MLP giảm params và tránh overfitting với data sparse.
    """
    def __init__(self, num_dense, embed_dim, hidden_dims):
        super().__init__()
        layers = []
        in_d = 1  # mỗi feature là scalar
        for h in hidden_dims:
            layers += [nn.Linear(in_d, h), nn.LayerNorm(h), nn.GELU()]
            in_d = h
        layers.append(nn.Linear(in_d, embed_dim))
        self.mlp = nn.Sequential(*layers)
        self.log_norm = Log1pNormalizer(1)  # normalize từng feature
        self.num_dense = num_dense

    def forward(self, x):
        # x: (B, 13)
        B = x.size(0)
        out = []
        for i in range(self.num_dense):
            xi = x[:, i:i+1]           # (B, 1)
            xi = self.log_norm(xi)      # normalize
            out.append(self.mlp(xi))    # (B, embed_dim)
        return torch.stack(out, dim=1)  # (B, 13, embed_dim)

print('Numeric processors defined.')

Numeric processors defined.


In [9]:
# ============================================================
# Cell 9 — DeepFM with DHE  (FIXED: BCEWithLogitsLoss, no sigmoid in model)
# ============================================================
class DeepFM_DHE(nn.Module):
    """
    DeepFM với DHE embedding cho categorical features.
    Input dense  : (B, 13) → log1p normalize → concat trực tiếp
    Input sparse : (B, 26) → DHE → (B, 26, 64)

    FIX so với bản gốc:
    - Bỏ sigmoid() ở output, dùng BCEWithLogitsLoss
    - FM terms được normalize trước khi cộng để tránh gradient explosion
    - LayerNorm trên FM output
    """
    def __init__(self, num_dense, vocab_sizes, embed_dim, hidden_dims,
                 dhe_num_hashes, dhe_hidden):
        super().__init__()
        self.num_dense = num_dense
        self.num_sparse = len(vocab_sizes)
        self.embed_dim = embed_dim

        # Dense normalization
        self.dense_norm = Log1pNormalizer(num_dense)

        # DHE for cat features
        self.dhe = DHELayer(vocab_sizes, embed_dim, dhe_num_hashes, dhe_hidden)

        # FM order-1: linear weights cho sparse (scalar per feature)
        # Không dùng Embedding cho FM-1 vì DHE đã encode, ta dùng linear projection
        self.fm1_sparse_proj = nn.Linear(embed_dim, 1, bias=False)  # shared
        self.fm1_dense_proj  = nn.Linear(num_dense, 1)

        # FM order-2 normalization
        self.fm2_norm = nn.LayerNorm(1)

        # Deep component
        deep_in = num_dense + self.num_sparse * embed_dim
        layers = []
        in_d = deep_in
        for dim in hidden_dims:
            layers += [nn.Linear(in_d, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.1)]
            in_d = dim
        layers.append(nn.Linear(in_d, 1))
        self.deep = nn.Sequential(*layers)

    def forward(self, dense_x, sparse_x):
        # Normalize dense
        dense_norm = self.dense_norm(dense_x)  # (B, 13)

        # DHE embeddings: (B, 26, 64)
        emb = self.dhe(sparse_x)  # (B, 26, 64)

        # --- FM Order-1 ---
        # Sparse: sum over features after linear proj
        fm1_sparse = self.fm1_sparse_proj(emb).squeeze(-1).sum(dim=1, keepdim=True)  # (B,1)
        fm1_dense  = self.fm1_dense_proj(dense_norm)                                  # (B,1)
        fm1 = fm1_sparse + fm1_dense

        # --- FM Order-2 ---
        # Chỉ tính trên sparse embeddings
        sum_emb    = emb.sum(dim=1)         # (B, 64)
        sum_sq_emb = sum_emb ** 2
        sq_sum_emb = (emb ** 2).sum(dim=1)
        fm2 = 0.5 * (sum_sq_emb - sq_sum_emb).sum(dim=1, keepdim=True)  # (B,1)
        fm2 = self.fm2_norm(fm2)  # normalize để tránh scale mismatch

        # --- Deep ---
        deep_in = torch.cat([dense_norm, emb.view(dense_x.size(0), -1)], dim=1)
        deep_out = self.deep(deep_in)  # (B,1)

        # Raw logit (NO sigmoid)
        return fm1 + fm2 + deep_out  # (B,1)

print('DeepFM_DHE defined.')

DeepFM_DHE defined.


In [10]:
# ============================================================
# Cell 10 — DCN with DHE  (FIXED)
# ============================================================
class CrossNetwork(nn.Module):
    """Cross Network — DCN-V1 style."""
    def __init__(self, input_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        self.w = nn.ParameterList([nn.Parameter(torch.empty(input_dim)) for _ in range(num_layers)])
        self.b = nn.ParameterList([nn.Parameter(torch.zeros(input_dim)) for _ in range(num_layers)])
        for w in self.w:
            nn.init.xavier_uniform_(w.unsqueeze(0))

    def forward(self, x0):
        xl = x0
        for i in range(self.num_layers):
            xlw = (xl * self.w[i]).sum(dim=1, keepdim=True)  # (B,1)
            xl  = x0 * xlw + self.b[i] + xl
        return xl


class DCN_DHE(nn.Module):
    """
    DCN với DHE embedding.
    Dense: log1p normalize (không embed thành 64 chiều)
    Sparse: DHE → 64 chiều
    Input dim = 13 + 26*64 = 1677

    FIX: bỏ sigmoid trong model, dùng BCEWithLogitsLoss
    """
    def __init__(self, num_dense, vocab_sizes, embed_dim, cross_layers, hidden_dims,
                 dhe_num_hashes, dhe_hidden):
        super().__init__()
        self.num_dense  = num_dense
        self.num_sparse = len(vocab_sizes)
        self.embed_dim  = embed_dim

        self.dense_norm = Log1pNormalizer(num_dense)
        self.dhe = DHELayer(vocab_sizes, embed_dim, dhe_num_hashes, dhe_hidden)

        input_dim = num_dense + self.num_sparse * embed_dim  # 13 + 26*64 = 1677
        self.cross_net = CrossNetwork(input_dim, cross_layers)

        deep_layers = []
        in_d = input_dim
        for dim in hidden_dims:
            deep_layers += [nn.Linear(in_d, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.1)]
            in_d = dim
        self.deep_net = nn.Sequential(*deep_layers)

        # Output: cross + deep → 1
        self.fc = nn.Linear(input_dim + in_d, 1)

    def forward(self, dense_x, sparse_x):
        dense_norm = self.dense_norm(dense_x)             # (B, 13)
        emb = self.dhe(sparse_x)                          # (B, 26, 64)
        emb_flat = emb.view(dense_x.size(0), -1)          # (B, 1664)

        x0 = torch.cat([dense_norm, emb_flat], dim=1)     # (B, 1677)

        cross_out = self.cross_net(x0)                    # (B, 1677)
        deep_out  = self.deep_net(x0)                     # (B, last_dim)

        combined = torch.cat([cross_out, deep_out], dim=1)
        return self.fc(combined)  # raw logit, NO sigmoid

print('DCN_DHE defined.')

DCN_DHE defined.


In [11]:
# ============================================================
# Cell 11 — DLRM with DHE
# ============================================================
class DLRM_DHE(nn.Module):
    """
    DLRM với DHE embedding.
    Dense features: MLP → 64-dim (Bottom MLP alternative với DenseEmbedMLP)
    Sparse features: DHE → 64-dim
    Tất cả features → 64-dim → Dot-product interactions → Top MLP
    """
    def __init__(self, num_dense, vocab_sizes, embed_dim,
                 dense_embed_hidden, top_mlp_dims,
                 dhe_num_hashes, dhe_hidden):
        super().__init__()
        self.num_dense  = num_dense
        self.num_sparse = len(vocab_sizes)
        self.embed_dim  = embed_dim

        # Dense: 13 scalar features → 13 vectors of dim 64
        self.dense_embed = DenseEmbedMLP(num_dense, embed_dim, dense_embed_hidden)

        # Sparse: DHE
        self.dhe = DHELayer(vocab_sizes, embed_dim, dhe_num_hashes, dhe_hidden)

        # Số features: 13 dense + 26 sparse = 39
        num_all = num_dense + self.num_sparse  # 39
        # Số dot-product pairs (upper triangle, offset=1)
        num_interactions = (num_all * (num_all - 1)) // 2
        # Top MLP input: 1 dense vec (embed_dim) + interactions
        # Theo paper DLRM, concat embed_dim (từ bottom MLP) + interactions
        # Ở đây ta dùng mean của dense embeds thay cho 1 bottom MLP
        top_in = embed_dim + num_interactions

        top_layers = []
        in_d = top_in
        for dim in top_mlp_dims:
            top_layers += [nn.Linear(in_d, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.1)]
            in_d = dim
        top_layers.append(nn.Linear(in_d, 1))
        self.top_mlp = nn.Sequential(*top_layers)

    def forward(self, dense_x, sparse_x):
        # Dense: (B, 13) → (B, 13, 64)
        dense_emb = self.dense_embed(dense_x)   # (B, 13, 64)
        # Sparse: (B, 26) → (B, 26, 64)
        sparse_emb = self.dhe(sparse_x)          # (B, 26, 64)

        # Concat all: (B, 39, 64)
        all_emb = torch.cat([dense_emb, sparse_emb], dim=1)  # (B, 39, 64)

        # Dot-product interactions
        Z = torch.bmm(all_emb, all_emb.transpose(1, 2))  # (B, 39, 39)
        n = all_emb.size(1)
        rows, cols = torch.triu_indices(n, n, offset=1)
        interactions = Z[:, rows, cols]  # (B, num_interactions)

        # Dùng mean của dense embeds làm "dense representative"
        dense_rep = dense_emb.mean(dim=1)  # (B, 64)

        concat = torch.cat([dense_rep, interactions], dim=1)  # (B, 64+num_int)
        return self.top_mlp(concat)  # raw logit

print('DLRM_DHE defined.')

DLRM_DHE defined.


In [12]:
# ============================================================
# Cell 12 — Training Engine
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()
    total_loss = 0
    for dense, cat, label in loader:
        dense, cat, label = dense.to(device), cat.to(device), label.to(device)
        optimizer.zero_grad()
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                logit = model(dense, cat).squeeze(1)
                loss  = criterion(logit, label)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logit = model(dense, cat).squeeze(1)
            loss  = criterion(logit, label)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total_loss += loss.item() * label.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    for dense, cat, label in loader:
        dense, cat, label = dense.to(device), cat.to(device), label.to(device)
        logit = model(dense, cat).squeeze(1)
        loss  = criterion(logit, label)
        total_loss += loss.item() * label.size(0)
        all_preds.append(torch.sigmoid(logit).cpu().numpy())
        all_labels.append(label.cpu().numpy())
    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    auc    = roc_auc_score(labels, preds)
    return total_loss / len(loader.dataset), auc


def run_experiment(model_name, model, cfg, train_dl, val_dl, test_dl, device):
    print(f'\n{'='*60}')
    print(f'  Model: {model_name}  |  Embedding: DHE')
    nparams = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Params: {nparams:,}')
    print(f'{'='*60}')

    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg['lr'], steps_per_epoch=len(train_dl), epochs=cfg['epochs']
    )
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    best_val_auc = 0
    results = []
    for epoch in range(1, cfg['epochs'] + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_dl, optimizer, criterion, device, scaler)
        scheduler.step()
        val_loss, val_auc = evaluate(model, val_dl, criterion, device)
        elapsed = time.time() - t0
        print(f'  Epoch {epoch}/{cfg["epochs"]} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | {elapsed:.1f}s')
        results.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'val_auc': val_auc})
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), f'{model_name}_DHE_best.pt')

    # Test evaluation
    model.load_state_dict(torch.load(f'{model_name}_DHE_best.pt'))
    test_loss, test_auc = evaluate(model, test_dl, criterion, device)
    print(f'\n  Test AUC: {test_auc:.4f}  |  Test Loss: {test_loss:.4f}')
    return results, test_auc

print('Training engine ready.')

Training engine ready.


In [13]:
# ============================================================
# Cell 13 — Run DeepFM + DHE
# ============================================================
deepfm_model = DeepFM_DHE(
    num_dense    = len(DENSE_COLS),
    vocab_sizes  = vocab_sizes,
    embed_dim    = CFG['embed_dim'],
    hidden_dims  = CFG['deepfm_hidden'],
    dhe_num_hashes = CFG['dhe_num_hash'],
    dhe_hidden   = CFG['dhe_hidden'],
)
deepfm_results, deepfm_test_auc = run_experiment(
    'DeepFM', deepfm_model, CFG, train_dl, val_dl, test_dl, DEVICE
)


  Model: DeepFM  |  Embedding: DHE
  Params: 18,521,819
  Epoch 1/5 | Train Loss: 0.4606 | Val Loss: 0.4510 | Val AUC: 0.7977 | 188.0s
  Epoch 2/5 | Train Loss: 0.4476 | Val Loss: 0.4471 | Val AUC: 0.8021 | 180.8s
  Epoch 3/5 | Train Loss: 0.4420 | Val Loss: 0.4458 | Val AUC: 0.8036 | 167.1s
  Epoch 4/5 | Train Loss: 0.4375 | Val Loss: 0.4461 | Val AUC: 0.8036 | 174.7s
  Epoch 5/5 | Train Loss: 0.4334 | Val Loss: 0.4470 | Val AUC: 0.8023 | 175.7s

  Test AUC: 0.8015  |  Test Loss: 0.4516


In [14]:
# ============================================================
# Cell 14 — Run DCN + DHE
# ============================================================
dcn_model = DCN_DHE(
    num_dense    = len(DENSE_COLS),
    vocab_sizes  = vocab_sizes,
    embed_dim    = CFG['embed_dim'],
    cross_layers = CFG['dcn_cross_layers'],
    hidden_dims  = CFG['dcn_hidden'],
    dhe_num_hashes = CFG['dhe_num_hash'],
    dhe_hidden   = CFG['dhe_hidden'],
)
dcn_results, dcn_test_auc = run_experiment(
    'DCN', dcn_model, CFG, train_dl, val_dl, test_dl, DEVICE
)


  Model: DCN  |  Embedding: DHE
  Params: 18,663,030
  Epoch 1/5 | Train Loss: 0.4587 | Val Loss: 0.4506 | Val AUC: 0.7984 | 175.0s
  Epoch 2/5 | Train Loss: 0.4467 | Val Loss: 0.4479 | Val AUC: 0.8024 | 179.1s
  Epoch 3/5 | Train Loss: 0.4413 | Val Loss: 0.4460 | Val AUC: 0.8036 | 172.1s
  Epoch 4/5 | Train Loss: 0.4369 | Val Loss: 0.4465 | Val AUC: 0.8032 | 173.2s
  Epoch 5/5 | Train Loss: 0.4328 | Val Loss: 0.4473 | Val AUC: 0.8023 | 173.4s

  Test AUC: 0.8014  |  Test Loss: 0.4519


In [ ]:
# ============================================================
# Cell 15 — Run DLRM + DHE
# ============================================================
dlrm_model = DLRM_DHE(
    num_dense          = len(DENSE_COLS),
    vocab_sizes        = vocab_sizes,
    embed_dim          = CFG['embed_dim'],
    dense_embed_hidden = CFG['int_mlp_dims'],
    top_mlp_dims       = CFG['dlrm_top_mlp'],
    dhe_num_hashes     = CFG['dhe_num_hash'],
    dhe_hidden         = CFG['dhe_hidden'],
)
dlrm_results, dlrm_test_auc = run_experiment(
    'DLRM', dlrm_model, CFG, train_dl, val_dl, test_dl, DEVICE
)


  Model: DLRM  |  Embedding: DHE
  Params: 18,114,435
  Epoch 1/5 | Train Loss: 0.4777 | Val Loss: 0.4711 | Val AUC: 0.7732 | 217.4s
  Epoch 2/5 | Train Loss: 0.4665 | Val Loss: 0.4692 | Val AUC: 0.7760 | 210.5s
  Epoch 3/5 | Train Loss: 0.4608 | Val Loss: 0.4691 | Val AUC: 0.7760 | 215.7s


In [ ]:
# ============================================================
# Cell 16 — Summary & Comparison
# ============================================================
print('\n' + '='*60)
print('  FINAL RESULTS — MHE Embedding')
print('='*60)
print(f'  DeepFM-MHE  Test AUC: {deepfm_test_auc:.4f}')
print(f'  DCN-MHE     Test AUC: {dcn_test_auc:.4f}')
print(f'  DLRM-MHE    Test AUC: {dlrm_test_auc:.4f}')
print('='*60)

summary = pd.DataFrame({
    'model'     : ['DeepFM-MHE', 'DCN-MHE', 'DLRM-MHE'],
    'embedding' : ['MHE'] * 3,
    'test_auc'  : [deepfm_test_auc, dcn_test_auc, dlrm_test_auc],
    'num_files' : [CFG['num_files']] * 3,
})


